# Heart Disease Prediction - Score Booster
### Ensemble of CatBoost + XGBoost + LightGBM with Feature Engineering
### Target: 0.95+ on Kaggle Playground Series S6E2

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path = '/kaggle/input/playground-series-s6e2/train.csv'
test_path  = '/kaggle/input/playground-series-s6e2/test.csv'

df     = pd.read_csv(train_path)
testdf = pd.read_csv(test_path)
print(f"Train: {df.shape}, Test: {testdf.shape}")

In [ ]:
def add_features(data):
    d = data.copy()

    # Interaction features (clinically meaningful)
    d['ST_x_Slope']         = d['ST depression'] * d['Slope of ST']
    d['Vessels_x_Thallium'] = d['Number of vessels fluro'] * d['Thallium']
    d['ST_x_ExAngina']      = d['ST depression'] * d['Exercise angina']
    d['BP_x_Chol']          = d['BP'] * d['Cholesterol']
    d['BP_x_Age']           = d['BP'] * d['Age']

    # Age-adjusted features
    d['MaxHR_per_Age']      = d['Max HR'] / (d['Age'] + 1)
    d['Chol_per_Age']       = d['Cholesterol'] / (d['Age'] + 1)

    # Cardiac stress score
    d['Cardiac_Stress'] = (
        d['ST depression'] * d['Exercise angina'] +
        d['Number of vessels fluro'] * d['Thallium']
    )

    # Heart rate reserve proxy
    d['HR_Reserve'] = (220 - d['Age']) - d['Max HR']

    # Risk combo flags
    d['High_Risk_Combo'] = (
        (d['Chest pain type'] == 4).astype(int) +
        (d['Exercise angina'] == 1).astype(int) +
        (d['Number of vessels fluro'] >= 1).astype(int)
    )

    return d

df     = add_features(df)
testdf = add_features(testdf)

X = df.drop(columns=['Heart Disease', 'id'])
y = df['Heart Disease']  # Keep as string labels
test_features = testdf.drop(columns=['id'])

print(f"Total features: {len(X.columns)}")
print(f"Features: {X.columns.tolist()}")

In [ ]:
N_FOLDS = 10

cat_params = {
    'iterations':          3000,
    'learning_rate':       0.03,
    'depth':               8,
    'l2_leaf_reg':         3,
    'bagging_temperature': 0.5,
    'random_strength':     0.5,
    'border_count':        128,
    'loss_function':       'Logloss',
    'eval_metric':         'Logloss',
    'random_state':        42,
    'allow_writing_files': False,
    'verbose':             False,
    'task_type':           'GPU',
}

xgb_params = {
    'n_estimators':     3000,
    'learning_rate':    0.03,
    'max_depth':        7,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'eval_metric':      'auc',
    'random_state':     42,
    'verbosity':        0,
    'tree_method':      'hist',
    'device':           'cuda',
}

lgbm_params = {
    'n_estimators':      3000,
    'learning_rate':     0.03,
    'num_leaves':        64,
    'max_depth':         7,
    'min_child_samples': 30,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'objective':         'binary',
    'metric':            'auc',
    'random_state':      42,
    'verbosity':         -1,
    # 'device':          'gpu',  # uncomment if LightGBM GPU works for you
}

print("Model configs ready. GPU enabled for CatBoost & XGBoost.")

In [ ]:
# Encode y for XGBoost/LightGBM (they need numeric)
y_numeric = y.map({'Absence': 0, 'Presence': 1})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_cat  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_lgbm = np.zeros(len(X))

test_cat  = np.zeros(len(testdf))
test_xgb  = np.zeros(len(testdf))
test_lgbm = np.zeros(len(testdf))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    y_tr_num, y_val_num = y_numeric.iloc[tr_idx], y_numeric.iloc[val_idx]

    # ---- CatBoost (uses string labels) ----
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_tr, y_tr)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_cat += cat_model.predict_proba(test_features)[:, 1] / N_FOLDS

    # ---- XGBoost (uses numeric labels) ----
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_tr, y_tr_num,
                  eval_set=[(X_val, y_val_num)],
                  verbose=False)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(test_features)[:, 1] / N_FOLDS

    # ---- LightGBM (uses numeric labels) ----
    lgbm_model = LGBMClassifier(**lgbm_params)
    lgbm_model.fit(X_tr, y_tr_num,
                   eval_set=[(X_val, y_val_num)])
    oof_lgbm[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]
    test_lgbm += lgbm_model.predict_proba(test_features)[:, 1] / N_FOLDS

    print(f"  Cat:  {roc_auc_score(y_val_num, oof_cat[val_idx]):.6f}")
    print(f"  XGB:  {roc_auc_score(y_val_num, oof_xgb[val_idx]):.6f}")
    print(f"  LGBM: {roc_auc_score(y_val_num, oof_lgbm[val_idx]):.6f}")

print("\n" + "="*50)
print(f"Overall OOF AUC - CatBoost: {roc_auc_score(y_numeric, oof_cat):.6f}")
print(f"Overall OOF AUC - XGBoost:  {roc_auc_score(y_numeric, oof_xgb):.6f}")
print(f"Overall OOF AUC - LightGBM: {roc_auc_score(y_numeric, oof_lgbm):.6f}")

In [ ]:
best_auc = 0
best_weights = (1, 0, 0)

for w_cat in np.arange(0.1, 0.9, 0.05):
    for w_xgb in np.arange(0.05, 0.9 - w_cat, 0.05):
        w_lgbm = 1.0 - w_cat - w_xgb
        if w_lgbm < 0.01:
            continue
        oof_blend = w_cat * oof_cat + w_xgb * oof_xgb + w_lgbm * oof_lgbm
        auc = roc_auc_score(y_numeric, oof_blend)
        if auc > best_auc:
            best_auc = auc
            best_weights = (w_cat, w_xgb, w_lgbm)

w_cat, w_xgb, w_lgbm = best_weights
print(f"Best Ensemble Weights:")
print(f"  CatBoost: {w_cat:.2f}")
print(f"  XGBoost:  {w_xgb:.2f}")
print(f"  LightGBM: {w_lgbm:.2f}")
print(f"  Ensemble OOF AUC: {best_auc:.6f}")

In [ ]:
final_preds = w_cat * test_cat + w_xgb * test_xgb + w_lgbm * test_lgbm

submission = pd.DataFrame({
    'id': testdf['id'],
    'Heart Disease': final_preds
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"submission.csv saved!")
print(submission.head(10))
print(f"\nPrediction stats: mean={final_preds.mean():.4f}, std={final_preds.std():.4f}")